# Overleaf Figures
Figures for the master thesis paper.

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize

from utils.mri.data_loader import DataLoader
from utils.mri.data_converter import DataConverter

root_path      = 'data/'
random_seed    = 42
# Crop margins used during training: removes noisy/empty borders
CROP = ((16, 10, 0), (17, 11, 17))

data_loader    = DataLoader(root_path=root_path)
data_converter = DataConverter()

FIG_DIR = 'out/overleaf_figures'
import os
os.makedirs(FIG_DIR, exist_ok=True)

def load_cropped_slice(path, slice=None):
    """Load NIfTI, apply training crop, return axial middle slice (H x W)."""
    vol = data_converter.load_path_as_numpy(path)   # (D, H, W)
    d0, h0, w0 = CROP[0]
    d1, h1, w1 = CROP[1]
    D, H, W = vol.shape
    vol = vol[d0:D-d1, h0:H-h1, w0:W-w1]
    if slice is None: slice = vol.shape[2] // 2
    return vol[:, :, slice]

random.seed(random_seed)
print('Imports OK')

---
## Background

### Figure 1 — Cropped HUNT brain with tissue legend
Shows WM (~1.0), GM (~0.8) and CSF (~0) intensity levels.

In [ ]:
random.seed(random_seed)
h3_path, _ = data_loader.get_random_pair_path()
slice1 = load_cropped_slice(h3_path)

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(slice1.T, cmap='gray', vmin=0, vmax=1, origin='lower')
ax.axis('off')

# Build a legend with three intensity swatches
wm_patch  = mpatches.Patch(facecolor=str(0.75),  edgecolor='gray', linewidth=0.5, label='WM')
gm_patch  = mpatches.Patch(facecolor=str(0.5),  edgecolor='gray', linewidth=0.5, label='GM')
csf_patch = mpatches.Patch(facecolor=str(0.05), edgecolor='gray', linewidth=0.5, label='CSF')

ax.legend(
    handles=[wm_patch, gm_patch, csf_patch],
    loc='lower right',
    fontsize=9,
    edgecolor='gray',
)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/bg_tissue_legend.png', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
random.seed(random_seed)
h3_path_seg, _ = data_loader.get_random_pair_path()   # same subject as Figure 1
seg_path = h3_path_seg.replace('_T1_PREP_MNI', '_SEG_3_PREP_MNI')

t1_slice  = load_cropped_slice(h3_path_seg)   # (H, W) float [0,1]
seg_slice = load_cropped_slice(seg_path)       # (H, W) int labels

wm_mask = (seg_slice == 3).T
gm_mask = (seg_slice == 2).T

# Size figure so each panel matches the slice aspect ratio exactly
H, W = t1_slice.T.shape
panel_w = 4.5
panel_h = panel_w * H / W
fig, axes = plt.subplots(1, 2, figsize=(panel_w * 2, panel_h),
                         gridspec_kw={'wspace': 0})

for ax in axes:
    ax.imshow(t1_slice.T, cmap='gray', vmin=0, vmax=1, origin='lower')
    ax.axis('off')

# Right panel: semi-transparent coloured overlays — two shades of blue
wm_rgba = np.zeros((*wm_mask.shape, 4))
gm_rgba = np.zeros((*gm_mask.shape, 4))
wm_rgba[wm_mask] = [0.55, 0.80, 1.00, 0.55]   # light blue
gm_rgba[gm_mask] = [0.08, 0.28, 0.75, 0.55]   # dark blue

axes[1].imshow(wm_rgba, origin='lower')
axes[1].imshow(gm_rgba, origin='lower')

wm_patch = mpatches.Patch(color=(0.55, 0.80, 1.00), label='White Matter')
gm_patch = mpatches.Patch(color=(0.08, 0.28, 0.75), label='Grey Matter')
axes[1].legend(handles=[wm_patch, gm_patch], loc='lower right', fontsize=9, edgecolor='gray')
axes[1].set_title('Segmented', fontsize=12, fontweight='bold')
axes[0].set_title('HUNT Scan',      fontsize=12, fontweight='bold')

plt.savefig(f'{FIG_DIR}/bg_wm_gm_seg_overlay.png', bbox_inches='tight', dpi=300)
plt.show()

### Figure 2 — Cropped HUNT brain (no legend) for lateral ventricle illustration

In [ ]:
# Pick a different subject to avoid repeating the same brain
all_pairs = data_loader.get_all_pair_paths()
random.seed(random_seed + 7)
h3_path2, _ = random.choice(all_pairs)
slice2 = load_cropped_slice(h3_path2, 100)

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(slice2.T, cmap='gray', vmin=0, vmax=1, origin='lower')
ax.axis('off')

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/bg_lateral_ventricles.png', bbox_inches='tight', dpi=300)
plt.show()

### Figure 3 — HUNT3 / HUNT4 pair and absolute difference
Shows longitudinal change between the two timepoints.

In [ ]:
random.seed(68)
h3_path3, h4_path3 = random.choice(all_pairs)
slice_h3 = load_cropped_slice(h3_path3, 100)
slice_h4 = load_cropped_slice(h4_path3, 100)
slice_diff = np.abs(slice_h4 - slice_h3)

fig, axes = plt.subplots(1, 3, figsize=(12, 4.5), gridspec_kw={'wspace': 0})

titles = ['HUNT3', 'HUNT4', 'Change']
slices = [slice_h3, slice_h4, slice_diff]
cmaps  = ['gray', 'gray', 'hot']
vmaxes = [1.0, 1.0, slice_diff.max() or 0.3]

for ax, title, sl, cmap, vmax in zip(axes, titles, slices, cmaps, vmaxes):
    im = ax.imshow(sl.T, cmap=cmap, vmin=0, vmax=vmax, origin='lower')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.axis('off')
    if cmap == 'hot':
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.01)
        cbar.set_label('|HUNT4 − HUNT3|', fontsize=9)

plt.savefig(f'{FIG_DIR}/bg_hunt34_pair_diff.png', bbox_inches='tight', dpi=300)
plt.show()

---
## Methodology

### Figure — Unprocessed vs HUNT3 preprocessed (same individual)
Left: raw T1 in native scanner space. Right: skull-stripped, MNI-registered, intensity-normalised HUNT3 output.

In [ ]:
import nibabel as nib
import utils.hunt_id_handler as h

# Old-format HUNT3 ID of the example subject for the methods figure. The
# original ID is withheld for privacy; replace with an ID from your own data.
old_id   = 9410000000000  # placeholder
short_id = h.old_to_short(old_id)

unproc_path = f"data/unprocessed/{old_id}/mri/T1.mgz"
proc_path   = f"data/HUNT3/{short_id}/{short_id}_0_T1_PREP_MNI.nii.gz"

slice_raw = 140  # axial slice index — adjust to taste
slice_mri = 92

# Reorient MGZ to RAS canonical so axis 2 is always S→I (axial top-down)
raw_img = nib.as_closest_canonical(nib.load(unproc_path))
raw_vol = raw_img.get_fdata()
mid_raw = raw_vol[:, :, slice_raw]
mid_raw = (mid_raw - mid_raw.min()) / (mid_raw.max() - mid_raw.min())

# Load processed — no crop
proc_vol = nib.load(proc_path).get_fdata()
mid_proc = proc_vol[:, :, slice_mri]
mid_proc = mid_proc[::-1, :]  # mirror left-right

fig, axes = plt.subplots(1, 2, figsize=(10, 5), gridspec_kw={'wspace': 0.02})
titles = ['HUNT3 Unprocessed', 'HUNT3 Preprocessed']
slices = [mid_raw, mid_proc]

for ax, sl, title in zip(axes, slices, titles):
    ax.imshow(sl.T, cmap='gray', vmin=0, vmax=1, origin='lower')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.axis('off')

plt.savefig(f'{FIG_DIR}/meth_unprocessed_vs_processed.png', bbox_inches='tight', dpi=300)
plt.show()

---
## Results

In [ ]:
# Results figures go here